<a href="https://colab.research.google.com/github/ram1o1/qwen3-tts-oneshot-cloning/blob/main/qwen3tts_oneshot_cloning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# STEP 0: CHECK FOR GPU BEFORE DOING ANYTHING
import torch

if not torch.cuda.is_available():
    print("🛑 WAIT! You don't have a GPU enabled. This will be too slow!")
    print("To fix this:")
    print("1. Go to the top menu and click 'Runtime' > 'Change runtime type'.")
    print("2. Under 'Hardware accelerator', select 'T4 GPU'.")
    print("3. Click 'Save' and run this cell again.")
    raise SystemError("No GPU detected. Please enable a GPU using the steps above.")

print("✅ GPU detected! Setting up your workspace...")


# STEP 1: INSTALL DEPENDENCIES
import os
print("⏳ Installing required packages (this takes about 1-2 minutes)...")
os.system("pip install -q -U qwen-tts gradio soundfile")
print("✅ Packages installed!")

# STEP 2: LOAD THE AI MODEL
import soundfile as sf
import gradio as gr
from qwen_tts import Qwen3TTSModel

print("⏳ Downloading the Qwen3-TTS AI Model (this might take a moment)...")
model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
    device_map="auto",
    dtype=torch.bfloat16
)
print("✅ Model loaded successfully!")

# STEP 3: DEFINE THE APP LOGIC & UI
def generate_clone(ref_audio_path, ref_transcript, target_text, target_language):
    if not ref_audio_path:
        return None, "⚠️ Please upload or record a reference audio."
    if not ref_transcript:
         return None, "⚠️ Please type the exact transcript of your reference audio."
    if not target_text:
        return None, "⚠️ Please type the new text you want the voice to say."

    try:
        # Generate the cloned voice
        wavs, sr = model.generate_voice_clone(
            text=target_text,
            language=target_language,
            ref_audio=ref_audio_path,
            ref_text=ref_transcript,
        )

        # Save output
        output_file = "my_cloned_voice.wav"
        sf.write(output_file, wavs[0], sr)
        return output_file, "🎉 Success! Your cloned voice is ready. Click play below!"
    except Exception as e:
        return None, f"❌ An error occurred: {str(e)}"

# Build the beautiful UI
with gr.Blocks(title="🎙️ Voice Cloner") as demo:

    # Header area
    gr.Markdown(
        """
        # 🎙️ Voice Cloner (Powered by Qwen3-TTS)
        Welcome! This tool lets you clone any voice in seconds. Just follow the 3 steps below.
        *Tip: For best results, use a clear, 3-to-5 second audio clip with no background noise.*

        """
    )

    with gr.Row():
        # LEFT COLUMN: Inputs
        with gr.Column(scale=2):

            # Step 1 Box
            with gr.Group():
                gr.Markdown("### 🛠️ Step 1: Provide the Reference Voice")
                ref_audio = gr.Audio(
                    label="Upload a short audio file OR record from your mic",
                    type="filepath"
                )
                ref_transcript = gr.Textbox(
                    label="What is being said in the audio above?",
                    placeholder="Type the exact words spoken in your recording...",
                    lines=2
                )

            gr.HTML("<br>") # Add a little space

            # Step 2 Box
            with gr.Group():
                gr.Markdown("### ✍️ Step 2: Write the New Script")
                target_text = gr.Textbox(
                    label="What should the voice say now?",
                    placeholder="Type the new sentence or paragraph here...",
                    lines=3
                )
                target_language = gr.Radio(
                    choices=["English", "Chinese", "Japanese", "Spanish", "French"],
                    value="English",
                    label="Target Language"
                )

            gr.HTML("<br>") # Add a little space

            # Step 3 Button
            generate_btn = gr.Button("✨ Step 3: Generate Voice ✨", variant="primary", size="lg")

        # RIGHT COLUMN: Outputs
        with gr.Column(scale=1):
            with gr.Group():
                gr.Markdown("### 🎧 Your Results")
                status_text = gr.Textbox(label="System Status", interactive=False, value="Waiting for input...")
                output_audio = gr.Audio(label="Generated Audio", interactive=False)

                gr.Markdown(
                    """
                    **How to download:** Once the audio generates, click the three dots (`⋮`) or the download icon `⬇️` inside the audio player above.
                    """
                )

    # Connect the UI to the AI
    generate_btn.click(
        fn=generate_clone,
        inputs=[ref_audio, ref_transcript, target_text, target_language],
        outputs=[output_audio, status_text]
    )

print("🚀 Launching the User Interface below...")
demo.launch(inline=True, share=True, quiet=True, theme=gr.themes.Soft(primary_hue="sky"), height=1000)